# 🧪 LAB 1: EDA, PHÂN RÃ CHUỖI THỜI GIAN & KIỂM ĐỊNH TÍNH DỪNG (STATIONARITY)
> **Bối cảnh nghiệp vụ**: Dự báo số lượng **Bug / Task phát sinh hàng ngày** trong các dự án phần mềm trên hệ thống **TaskPilot**.
> **Mục tiêu kỹ thuật**: Nắm vững trực giác toán học và kỹ thuật phân tích chuỗi thời gian từ cơ bản đến kiểm định tính dừng (ADF & KPSS) trước khi xây dựng mô hình dự báo (SARIMAX, LightGBM, SOTA Transformers).

---

## 📌 Cấu trúc bài Lab:
1. **BƯỚC 1**: Khám phá dữ liệu (EDA) & Phân tích chu kỳ tuần (Day of Week Analysis).
2. **BƯỚC 2**: Phân rã chuỗi (Seasonal Decomposition): Additive ($Y = T + S + R$) vs Multiplicative ($Y = T \times S \times R$).
3. **BƯỚC 3**: Phân tích tương quan tự hồi quy qua biểu đồ **ACF** và **PACF**.
4. **BƯỚC 4**: Kiểm định tính dừng thống kê (**ADF Test** & **KPSS Test**).
5. **BƯỚC 5**: Kỹ thuật biến đổi và sai phân làm dừng chuỗi (Differencing & Log Transform).

### 1. Khởi tạo môi trường và Import các thư viện phân tích

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Thiết lập giao diện đồ thị chuyên nghiệp
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print('Đã import đầy đủ các thư viện phân tích chuỗi thời gian!')

### 2. Tải tập dữ liệu `daily_bug_reports.csv` (TaskPilot)
Chuỗi thời gian ghi nhận số lượng Bug phát sinh hàng ngày trong giai đoạn 3 năm liên tục (2021 - 2023).

In [ ]:
df = pd.read_csv('daily_bug_reports.csv', parse_dates=['date'], index_col='date')
print(f"Khoảng thời gian: {df.index.min().strftime('%d/%m/%Y')} -> {df.index.max().strftime('%d/%m/%Y')} ({len(df)} ngày quan sát)")
df.head(10)

In [ ]:
# Thống kê mô tả số lượng Bug
print("--- THỐNG KÊ MÔ TẢ (DESCRIPTIVE STATISTICS) ---")
print(df['bug_count'].describe())

# Trực quan hóa chuỗi thời gian và các đường trung bình trượt (Rolling Mean)
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df.index, df['bug_count'], label='Số Bug hàng ngày (Raw)', color='#94a3b8', alpha=0.55, linewidth=1)
ax.plot(df.index, df['bug_count'].rolling(window=7).mean(), label='Trung bình trượt 7 ngày (7-day MA)', color='#2563eb', linewidth=2)
ax.plot(df.index, df['bug_count'].rolling(window=30).mean(), label='Trung bình trượt 30 ngày (30-day MA)', color='#dc2626', linewidth=2)
ax.set_title('Chuỗi thời gian Bug phát sinh hàng ngày tại TaskPilot (2021 - 2023)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Thời gian', fontsize=11)
ax.set_ylabel('Số lượng Bug', fontsize=11)
ax.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

### 3. Phân tích tính mùa vụ theo các ngày trong tuần (Day of Week)
Đặc thù kỹ thuật phần mềm: Đội ngũ Dev/QA làm việc mạnh vào Thứ Hai - Thứ Năm, giảm vào Thứ Sáu và chạm đáy vào cuối tuần (Thứ Bảy, Chủ Nhật).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Boxplot phân phối Bug theo ngày
sns.boxplot(x='day_name', y='bug_count', data=df, order=day_order, ax=axes[0], hue='day_name', palette='Blues_r', legend=False)
axes[0].set_title('Phân phối số lượng Bug theo ngày trong tuần', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Cột trung bình theo ngày
day_mean = df.groupby('day_name')['bug_count'].mean().reindex(day_order)
axes[1].bar(day_order, day_mean, color=['#2563eb' if d not in ['Saturday', 'Sunday'] else '#f87171' for d in day_order], edgecolor='#1e293b', alpha=0.85)
axes[1].set_title('Trung bình số Bug phát sinh theo ngày', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(day_mean):
    axes[1].text(i, v + 0.3, f"{v:.1f}", ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 2. Phân rã chuỗi thời gian (Seasonal Decomposition: s=7 ngày)

Phân tách chuỗi $y_t$ thành 3 thành phần độc lập:
- **Trend ($T_t$)**: Xu hướng tăng giảm dài hạn của dự án.
- **Seasonality ($S_t$)**: Mẫu hình lặp lại đều đặn mỗi chu kỳ $s=7$ ngày.
- **Residual ($R_t$)**: Nhiễu ngẫu nhiên / đột biến (spikes) ngày Release.

Hai dạng mô hình:
1. **Mô hình Cộng (Additive)**: $y_t = T_t + S_t + R_t$ (khi biên độ dao động mùa vụ không đổi theo thời gian).
2. **Mô hình Nhân (Multiplicative)**: $y_t = T_t \times S_t \times R_t$ (khi biên độ mùa vụ tăng tỷ lệ thuận với mức nền Trend).

In [ ]:
# 1. Additive Model
series = df['bug_count']
decomp_add = seasonal_decompose(series, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(15, 9), sharex=True)
decomp_add.observed.plot(ax=axes[0], color='#0f172a', title='Observed (Thực tế)')
decomp_add.trend.plot(ax=axes[1], color='#2563eb', title='Trend (Xu hướng dài hạn)')
decomp_add.seasonal.plot(ax=axes[2], color='#16a34a', title='Seasonality (Chu kỳ mùa vụ tuần s=7)')
decomp_add.resid.plot(ax=axes[3], color='#dc2626', style='.', title='Residual (Nhiễu ngẫu nhiên)')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5)
fig.suptitle('Phân rã mô hình Cộng (Additive Decomposition): Y = Trend + Seasonal + Residual', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Phân tích tương quan (ACF & PACF)
- **ACF (Autocorrelation Function)**: Đo lường mức độ tương quan tuyến tính giữa $y_t$ và $y_{t-k}$. Đỉnh nhọn tại các bội số $k = 7, 14, 21, 28$ là bằng chứng không thể chối cãi của **tính mùa vụ tuần mạnh**.
- **PACF (Partial Autocorrelation Function)**: Đo lường tương quan giữa $y_t$ và $y_{t-k}$ sau khi đã loại bỏ ảnh hưởng của tất cả các độ trễ trung gian $y_{t-1}, \dots, y_{t-k+1}$. Giúp xác định bậc tự hồi quy $p$ cho mô hình AR.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_acf(series, lags=35, ax=axes[0], title='Autocorrelation Function (ACF) - Chuỗi gốc', color='#2563eb')
plot_pacf(series, lags=35, ax=axes[1], title='Partial Autocorrelation Function (PACF) - Chuỗi gốc', color='#16a34a', method='ywm')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5)
    for lag in [7, 14, 21, 28]:
        ax.axvline(x=lag, color='#f59e0b', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()

## 4. Kiểm định tính dừng (Stationarity Tests: ADF vs KPSS)

Một chuỗi thời gian dừng (Stationary) khi các thuộc tính thống kê của nó không đổi theo thời gian:
1. Kỳ vọng không đổi: $\mathbb{E}[y_t] = \mu$
2. Phương sai không đổi: $\text{Var}(y_t) = \sigma^2$
3. Hiệp phương sai tự tương quan chỉ phụ thuộc vào độ trễ $k$: $\text{Cov}(y_t, y_{t-k}) = \gamma_k$

| Kiểm định | Giả thuyết vô hiệu ($H_0$) | Kết luận khi $p < 0.05$ | Kết luận khi $p \ge 0.05$ |
| :--- | :--- | :--- | :--- |
| **ADF (Dickey-Fuller)** | Chuỗi có nghiệm đơn vị (Unit Root / Không dừng) | Bác bỏ $H_0 \rightarrow$ **DỪNG** | Không bác bỏ $H_0 \rightarrow$ **KHÔNG DỪNG** |
| **KPSS** | Chuỗi dừng quanh xu hướng / mức nền | Bác bỏ $H_0 \rightarrow$ **KHÔNG DỪNG** | Không bác bỏ $H_0 \rightarrow$ **DỪNG** |

> **Nguyên tắc kết hợp ADF & KPSS**:
> - ADF dừng + KPSS dừng: **Chuỗi hoàn toàn dừng (Stationary)**.
> - ADF không dừng + KPSS không dừng: **Chuỗi không dừng (Non-Stationary)**.
> - ADF dừng + KPSS không dừng: **Difference Stationary** (còn thành phần chu kỳ hoặc xu hướng cục bộ, cần sai phân).

In [ ]:
def test_stationarity(ts, name="Chuỗi"):
    clean = ts.dropna()
    print(f"=== KẾT QUẢ KIỂM ĐỊNH TÍNH DỪNG: {name.upper()} ===")
    
    # 1. ADF
    adf = adfuller(clean, autolag='AIC')
    print(f"1. ADF Test : Statistic = {adf[0]:.4f}, p-value = {adf[1]:.4e}")
    print(f"   Critical Values: 1%: {adf[4]['1%']:.3f}, 5%: {adf[4]['5%']:.3f}")
    adf_stat = adf[1] < 0.05
    print(f"   => ADF kết luận: {'CHUỖI DỪNG (p < 0.05)' if adf_stat else 'CHUỖI CHƯA DỪNG (p >= 0.05)'}")
    
    # 2. KPSS
    kp = kpss(clean, regression='c', nlags='auto')
    print(f"\n2. KPSS Test: Statistic = {kp[0]:.4f}, p-value = {kp[1]:.4f}")
    print(f"   Critical Values: 1%: {kp[3]['1%']:.3f}, 5%: {kp[3]['5%']:.3f}")
    kp_stat = kp[1] > 0.05
    print(f"   => KPSS kết luận: {'CHUỖI DỪNG (p > 0.05)' if kp_stat else 'CHUỖI CHƯA DỪNG (p <= 0.05)'}")
    
    # Tổng hợp
    print("\n=> KẾT LUẬN TỔNG HỢP (ADF + KPSS):")
    if adf_stat and kp_stat:
        verdict = "CHUỖI HOÀN TOÀN DỪNG (STATIONARY)"
    elif not adf_stat and not kp_stat:
        verdict = "CHUỖI KHÔNG DỪNG (NON-STATIONARY)"
    elif adf_stat and not kp_stat:
        verdict = "DIFFERENCE STATIONARY (Cần sai phân thêm để triệt tiêu tính chu kỳ)"
    else:
        verdict = "TREND STATIONARY (Dừng quanh xu hướng deterministic)"
    print(f"   >>> {verdict} <<<")
    print("-" * 60)

# Chạy kiểm định trên chuỗi gốc
test_stationarity(series, name="Chuỗi số lượng Bug gốc")

## 5. Biến đổi và Sai phân làm dừng chuỗi (Differencing Transformations)

Để chuỗi đạt tính dừng chuẩn mực phục vụ cho mô hình SARIMA ở Lab 2, ta áp dụng:
1. **Sai phân bậc 1 ($d=1$)**: $\Delta y_t = y_t - y_{t-1}$ để khử xu hướng (Trend).
2. **Sai phân mùa vụ tuần ($D=1, s=7$)**: $\Delta_7 y_t = y_t - y_{t-7}$ để khử dao động tuần.
3. **Sai phân kết hợp**: $\Delta(\Delta_7 y_t) = (y_t - y_{t-7}) - (y_{t-1} - y_{t-8})$

In [ ]:
diff_seasonal = series.diff(7)
diff_both = series.diff(7).diff(1)

fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)
series.plot(ax=axes[0], color='#0f172a', title='(a) Chuỗi gốc (Original)')
diff_seasonal.plot(ax=axes[1], color='#16a34a', title='(b) Sai phân mùa vụ tuần: Seasonal Diff (s=7)')
diff_both.plot(ax=axes[2], color='#2563eb', title='(c) Sai phân kết hợp: Diff(7) rồi Diff(1) [SARIMA ready]')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.axhline(0, color='#64748b', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

### 6. Kiểm định lại tính dừng sau khi sai phân kết hợp
Kiểm tra xem chuỗi $\Delta(\Delta_7 y_t)$ đã đạt tính dừng hoàn toàn theo cả 2 tiêu chí ADF và KPSS chưa.

In [ ]:
test_stationarity(diff_both, name="Chuỗi sau sai phân kết hợp Diff(7) + Diff(1)")

### 7. Biểu đồ ACF & PACF của chuỗi đã làm dừng
Quan sát biểu đồ sau sai phân để xác định các siêu tham số cho mô hình **SARIMA $(p,d,q)(P,D,Q)_7$**:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_acf(diff_both.dropna(), lags=35, ax=axes[0], title='ACF - Chuỗi đã làm dừng [Diff(7)+Diff(1)]', color='#2563eb')
plot_pacf(diff_both.dropna(), lags=35, ax=axes[1], title='PACF - Chuỗi đã làm dừng [Diff(7)+Diff(1)]', color='#16a34a', method='ywm')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 🎯 TỔNG KẾT LAB 1 & BƯỚC ĐỆM CHO LAB 2

1. **Đặc thù chuỗi Bug TaskPilot**:
   - Có tính mùa vụ theo tuần ($s=7$) rất rõ ràng: Đỉnh điểm phát sinh bug rơi vào các ngày làm việc (Thứ Hai/Thứ Ba), chạm đáy vào cuối tuần.
2. **Kết quả kiểm định tính dừng**:
   - Chuỗi gốc là **Difference Stationary** (chưa dừng hoàn toàn do yếu tố mùa vụ).
   - Áp dụng sai phân kết hợp mùa vụ $D=1, s=7$ và sai phân bậc nhất $d=1$ đã đưa chuỗi về **Stationary hoàn hảo** ($p_{\text{ADF}} < 10^{-20}$ và $p_{\text{KPSS}} > 0.05$).
3. **Chuẩn bị cho Lab 2**:
   - Với $d=1, D=1, s=7$, dữ liệu đã sẵn sàng để đưa vào huấn luyện các mô hình thống kê cổ điển **ARIMA** và **SARIMAX** có biến ngoại sinh (Flash Sale, Release Day).